In [1]:
from main import *
#from sympy import *

In [4]:
# setup sympy symbols
R = CoordSys3D("")
delop = Del()  # nabla
phi = sp.Function("phi")
eta = sp.Function('eta')
max_order = 1  # order of expansion
E_symbols, H_symbols, E_vec_comps, H_vec_comps, E_asympt, H_asympt = (
    gen_vector_field_symbols(R, order=max_order)
)

# contsruct Maxwell's equations
eqs_0, alg_eqs_0, diff_eqs_0 = gen_maxwell_eqs(
    R,
    delop,
    phi,
    E_symbols,
    H_symbols,
    E_vec_comps,
    H_vec_comps,
    E_asympt,
    H_asympt,
    order=0,
)

# eqs_1, alg_eqs_1, diff_eqs_1 = gen_maxwell_eqs(
#     R,
#     delop,
#     phi,
#     E_symbols,
#     H_symbols,
#     E_vec_comps,
#     H_vec_comps,
#     E_asympt,
#     H_asympt,
#     order=1,
#     prev_order_eqs=eqs_0,
# )

vars_subs_0 = gen_vars_subs([E_symbols[0], H_symbols[0]])
vars_subs_1 = gen_vars_subs([E_symbols[1], H_symbols[1]])

# solve ODE system
diff_sols_0 = dsolve_system(
    list_subs(diff_eqs_0, vars_subs_0),
    funcs=[
        E_symbols[0][1](x),
        E_symbols[0][2](x),
        H_symbols[0][1](x),
        H_symbols[0][2](x),
    ],
    t=x,
)[0]
assert check_sols(diff_eqs_0, diff_sols_0) == [True, True, True, True]

sols_0 = [
    alg_eqs_0[0].subs({eq.lhs.func(x, y, z): eq.rhs for eq in diff_sols_0}),
    diff_sols_0[0],
    diff_sols_0[1],
    alg_eqs_0[1].subs({eq.lhs.func(x, y, z): eq.rhs for eq in diff_sols_0}),
    diff_sols_0[2],
    diff_sols_0[3],
]
sols_0 = list_subs(sols_0, {sp.sqrt(-epsilon*mu+sp.diff(phi(z),z)**2):eta(z)}, eval=True)
# sols_1 = dsolve_system(
#     list_subs(diff_eqs_1, vars_subs_1),
#     funcs=[
#         E_symbols[1][1](x),
#         E_symbols[1][2](x),
#         H_symbols[1][1](x),
#         H_symbols[1][2](x),
#     ],
#     t=x,
# )[0]
# assert check_sols(diff_eqs_1, sols_1) == [True, True, True, True]

# preview(sols_1[0], output="png", dvioptions=["-D 600"], euler=False)
# preview(sols_1[1], output="png", dvioptions=["-D 600"], euler=False)
# preview(sols_1[2], output="png", dvioptions=["-D 600"], euler=False)
# preview(sols_1[3], output="png", dvioptions=["-D 600"], euler=False)

# Construct solutions for different layers
sols_0_layers = {
    layer: layered_sols(sols_0, layer, order=0) for layer in ["c", "f", "s"]
}

# Now solving for 2D waveguide with smoothly irregular transition, x=h(z)

# boundry conditions
h = sp.Function("h")
border_func = R.x - h(R.z)
E_boundry_cf = gen_boundry_conds(
    R, delop, E_vec_comps[0], border_func, h(z), ["c", "f"], order=0
)

H_boundry_cf = gen_boundry_conds(
    R, delop, H_vec_comps[0], border_func, h(z), ["c", "f"], order=0
)

E_boundry_fs = gen_boundry_conds(
    R, delop, E_vec_comps[0], border_func, 0, ["f", "s"], order=0
)

H_boundry_fs = gen_boundry_conds(
    R, delop, H_vec_comps[0], border_func, 0, ["f", "s"], order=0
)

boundry_eqs = list_subs(
    E_boundry_cf + H_boundry_cf + E_boundry_fs + H_boundry_fs,
    {eq.lhs: eq.rhs for eq in list_subs(sols_0_layers['c']+sols_0_layers['f'], {x:h(z)}, eval=False)
     +list_subs(sols_0_layers['f']+sols_0_layers['s'], {x:0}, eval=False)},
)
coeffs = [
    sp.Symbol(f"A_0^c"),
    sp.Symbol(f"B_0^c"),
    sp.Symbol(f"A_0^f"),
    sp.Symbol(f"B_0^f"),
    sp.Symbol(f"C_0^f"),
    sp.Symbol(f"D_0^f"),
    sp.Symbol(f"A_0^s"),
    sp.Symbol(f"B_0^s"),
]

In [9]:
boundry_eqs[4]

Eq((B_0^s*epsilon_f*(-I*eta_s(z) + Derivative(h(z), z)*Derivative(phi(z), z)) + epsilon_s*(-I*C_0^f*eta_f(z) - C_0^f*Derivative(h(z), z)*Derivative(phi(z), z) + I*D_0^f*eta_f(z) - D_0^f*Derivative(h(z), z)*Derivative(phi(z), z)))/(epsilon_f*epsilon_s), 0)

In [70]:
#boundry_eqs[3]
# TE: 1 2 5 6
# TM: 0 3 4 7
E_boundry_cf + H_boundry_cf + E_boundry_fs + H_boundry_fs

[Eq(-E^{x,c}_0(h(z))*Derivative(h(z), z) + E^{x,f}_0(h(z))*Derivative(h(z), z) - E^{z,c}_0(h(z)) + E^{z,f}_0(h(z)), 0),
 Eq(E^{y,c}_0(h(z)) - E^{y,f}_0(h(z)), 0),
 Eq(-H^{x,c}_0(h(z))*Derivative(h(z), z) + H^{x,f}_0(h(z))*Derivative(h(z), z) - H^{z,c}_0(h(z)) + H^{z,f}_0(h(z)), 0),
 Eq(H^{y,c}_0(h(z)) - H^{y,f}_0(h(z)), 0),
 Eq(-E^{x,f}_0(0)*Derivative(h(z), z) + E^{x,s}_0(0)*Derivative(h(z), z) - E^{z,f}_0(0) + E^{z,s}_0(0), 0),
 Eq(E^{y,f}_0(0) - E^{y,s}_0(0), 0),
 Eq(-H^{x,f}_0(0)*Derivative(h(z), z) + H^{x,s}_0(0)*Derivative(h(z), z) - H^{z,f}_0(0) + H^{z,s}_0(0), 0),
 Eq(H^{y,f}_0(0) - H^{y,s}_0(0), 0)]

In [10]:
new_ord = [1,2,5,6,0,3,4,7]
boundry_eqs = [boundry_eqs[i] for i in new_ord]

In [11]:
coeffs = [
    # TM
    sp.Symbol(f"A_0^c"),
    sp.Symbol(f"A_0^f"),
    sp.Symbol(f"B_0^f"),
    sp.Symbol(f"A_0^s"),
    # TE
    sp.Symbol(f"B_0^c"),
    sp.Symbol(f"C_0^f"),
    sp.Symbol(f"D_0^f"),
    sp.Symbol(f"B_0^s"),

]
M, b = sp.linear_eq_to_matrix(boundry_eqs, coeffs)
M

Matrix([
[                                                                    I*mu_c*exp(-omega*eta_c(z)*h(z)/c)/eta_c(z),                                                                   -I*mu_f*exp(-omega*eta_f(z)*h(z)/c)/eta_f(z),                                                                   I*mu_f*exp(omega*eta_f(z)*h(z)/c)/eta_f(z),                                                                                              0,                                                                                                                                       0,                                                                                                                                      0,                                                                                                                                                      0,                                                                   0],
[-exp(-omega*eta_c(z)*h(z)/c) + I*exp(-omega*eta_c(z)*h(z)/c)*De

In [12]:
M_TM = M[0:4,0:4]
coeffs_TE = coeffs[4:]
M_TE = M[4:,4:]
coeffs_TM = coeffs[:4]

In [13]:
M_TE.nullspace()

[]

In [14]:
sp.solve(boundry_eqs[:4], coeffs_TM)

{A_0^c: 0, A_0^f: 0, A_0^s: 0, B_0^f: 0}

In [16]:
sp.solve(x**2+y**2-1, [x, y])

[(-sqrt(1 - y**2), y), (sqrt(1 - y**2), y)]

In [17]:
boundry_eqs[0]

Eq(I*A_0^c*mu_c*exp(-omega*eta_c(z)*h(z)/c)/eta_c(z) - I*A_0^f*mu_f*exp(-omega*eta_f(z)*h(z)/c)/eta_f(z) + I*B_0^f*mu_f*exp(omega*eta_f(z)*h(z)/c)/eta_f(z), 0)